<a href="https://colab.research.google.com/github/AmiraFaisal/ETEC2T/blob/main/populate_evalSummary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Populate automatic metrics

In [ ]:
!pip install -q gspread google-auth pandas

ERROR: Operation cancelled by user


In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)


Mounted at /content/drive


In [ ]:
import os
EVAL_ROOT = '/content/drive/MyDrive/EvaltheEvaluators/evaluations'
SHEET_NAME = 'evalSummary'
SHEET_ID = '1fmOrs-cn9iM1z4lg76CCk9pq8fOKl-O0SMeYAxedxoM'
WORKSHEET_NAME = 'Sheet1'
CSV_MAP = {
    'Factuality/BERTScore-F1_results.csv' : 'BERTScore',
    'Factuality/ROUGE-1_results.csv'       : 'ROUGE',
    'Coherence/BARTScore_results.csv'      : 'BARTScore',
    'Fluency/PPL_results.csv'              : 'PPL',
    'Fluency/SPICE_results.csv'            : 'SPICE',
}

preview CSVs

In [ ]:
import pandas as pd

def load_csv(rel_path: str) -> pd.DataFrame:
    """Load a results CSV and normalise to columns: [chart_id, Qwen, ChartGemma]."""
    full_path = os.path.join(EVAL_ROOT, rel_path)
    df = pd.read_csv(full_path)
    cols = df.columns.tolist()

    # chart-id column: whichever is NOT Qwen / ChartGemma
    id_col = next(
        c for c in cols
        if not any(kw in c.lower() for kw in ['qwen', 'gemma', 'chartgemma'])
    )
    qwen_col = next(c for c in cols if 'qwen' in c.lower())
    gemma_col = next(c for c in cols if 'gemma' in c.lower())

    df = df.rename(columns={
        id_col   : 'chart_id',
        qwen_col : 'Qwen',
        gemma_col: 'ChartGemma',
    })[['chart_id', 'Qwen', 'ChartGemma']]

    df['chart_id'] = df['chart_id'].astype(str)
    return df

# Preview all CSVs
dfs = {}
for rel_path, metric in CSV_MAP.items():
    try:
        dfs[metric] = load_csv(rel_path)
        print(f'{metric:15s}  rows={len(dfs[metric])}  cols={dfs[metric].columns.tolist()}')
        display(dfs[metric].head(3))
    except Exception as e:
        print(f'{metric}: {e}')

BERTScore        rows=21  cols=['chart_id', 'Qwen', 'ChartGemma']


,chart_id,Qwen,ChartGemma
0,1061,-0.575951,0.277797
1,1064,0.193021,2.716564
2,1253,1.498477,0.797433


ROUGE            rows=21  cols=['chart_id', 'Qwen', 'ChartGemma']


,chart_id,Qwen,ChartGemma
0,1061,0.571429,0.263158
1,1064,1.904762,2.682927
2,1253,2.142857,1.458333


BARTScore        rows=21  cols=['chart_id', 'Qwen', 'ChartGemma']


,chart_id,Qwen,ChartGemma
0,1061,0.143061,0.134605
1,1064,0.079652,0.082294
2,1253,0.119052,0.085358


PPL              rows=21  cols=['chart_id', 'Qwen', 'ChartGemma']


,chart_id,Qwen,ChartGemma
0,1061,8.736341,6.104955
1,1064,10.000000,5.636746
2,1253,5.278006,7.227569


SPICE            rows=21  cols=['chart_id', 'Qwen', 'ChartGemma']


,chart_id,Qwen,ChartGemma
0,1061,1.000000,1.000000
1,1064,5.878151,5.074324
2,1253,3.571429,3.093750


check evalSummary

In [ ]:
ss  = gc.open(SHEET_NAME)
ws  = ss.worksheet(WORKSHEET_NAME)

all_values = ws.get_all_values()

print(f'Sheet dimensions: {len(all_values)} rows × {len(all_values[0])} cols')
print('\nFirst 4 rows (first 16 cols):')
for i, row in enumerate(all_values[:4]):
    print(f'  row {i}: {row[:16]}')

Sheet dimensions: 15 rows × 44 cols

First 4 rows (first 16 cols):
  row 0: ['Chart ID', '', '', '1', '1', '2', '2', '3', '3', '4', '4', '5', '5', '6', '6', '7']
  row 1: ['Metric/ Model', '', '', 'Qwen', 'Gemma', 'Qwen', 'Gemma', 'Qwen', 'Gemma', 'Qwen', 'Gemma', 'Qwen', 'Gemma', 'Qwen', 'Gemma', 'Gemma']
  row 2: ['Automatic', 'Factuality', 'BERTScore', '2.41', '1.98', '3.05', '4.55', '-0.58', '0.28', '0.19', '2.72', '1.50', '0.80', '2.71', '3.88', '1.36']
  row 3: ['', '', 'ROUGE', '2.63', '1.33', '2.20', '2.63', '0.57', '0.26', '1.90', '2.68', '2.14', '1.46', '2.62', '3.29', '2.11']


In [ ]:
all_chart_ids = sorted(
    set().union(*[set(df['chart_id'].tolist()) for df in dfs.values()]),
    key=lambda x: int(x) if x.isdigit() else x
)

# Maps e.g. '1061' → '1', '1064' → '2', etc.
real_id_to_column = {real: str(i + 1) for i, real in enumerate(all_chart_ids)}

print(f"Found {len(all_chart_ids)} unique chart IDs across all CSVs.")
print("\nMapping (real ID → sheet column):")
for real, column in real_id_to_column.items():
    print(f"  {real} → Chart ID {column}")

Found 21 unique chart IDs across all CSVs.

Mapping (real ID → sheet column):
  623 → Chart ID 1
  749 → Chart ID 2
  1061 → Chart ID 3
  1064 → Chart ID 4
  1253 → Chart ID 5
  1356 → Chart ID 6
  2114 → Chart ID 7
  2121 → Chart ID 8
  2264 → Chart ID 9
  2369 → Chart ID 10
  2472 → Chart ID 11
  3128 → Chart ID 12
  3190 → Chart ID 13
  4067 → Chart ID 14
  4845 → Chart ID 15
  4915 → Chart ID 16
  5932 → Chart ID 17
  7127 → Chart ID 18
  7567 → Chart ID 19
  8038 → Chart ID 20
  8673 → Chart ID 21


parse

In [ ]:
import re

header_row   = all_values[0]   # Chart IDs
model_row    = all_values[1]   # Qwen / Gemma labels

# Build: chart_id (str) → {Qwen: col_index, ChartGemma: col_index}
chart_col_map = {}   # '1061' -> {'Qwen': 3, 'ChartGemma': 4}

current_chart = None
for col_idx, (h, m) in enumerate(zip(header_row, model_row)):
    h = str(h).strip()
    m = str(m).strip()

    # New chart ID?
    if re.match(r'^\d+$', h):          # numeric chart id
        current_chart = h

    if current_chart is None:
        continue

    if current_chart not in chart_col_map:
        chart_col_map[current_chart] = {}

    if 'qwen' in m.lower():
        chart_col_map[current_chart]['Qwen'] = col_idx
    elif 'gemma' in m.lower():
        chart_col_map[current_chart]['ChartGemma'] = col_idx

print(f'Found {len(chart_col_map)} charts in sheet header.')
sample = list(chart_col_map.items())[:3]
for cid, cols in sample:
    print(f'  chart {cid}: {cols}')

Found 21 charts in sheet header.
  chart 1: {'Qwen': 3, 'ChartGemma': 4}
  chart 2: {'Qwen': 5, 'ChartGemma': 6}
  chart 3: {'Qwen': 7, 'ChartGemma': 8}


In [ ]:
# Build: metric label → row_index in the sheet
# The metric name appears in column C (index 2) of each data row.
metric_row_map = {}   # 'BERTScore' -> 2

for row_idx, row in enumerate(all_values):
    cell = str(row[2]).strip() if len(row) > 2 else ''
    if cell and cell not in ('Metric/ Model', ''):
        metric_row_map[cell] = row_idx

print('Metric → sheet row:')
for m, r in metric_row_map.items():
    print(f'  {m!r:20s}  → row {r}')

Metric → sheet row:
  'BERTScore'           → row 2
  'ROUGE'               → row 3
  'FC*'                 → row 4
  'BARTScore'           → row 5
  'Relevance*'          → row 6
  'PPL'                 → row 7
  'SPICE'               → row 8
  'Fluency*'            → row 9
  'Factuality'          → row 10
  'Coherence'           → row 11
  'Fluency'             → row 12


write

In [ ]:
import time

def col_letter(col_idx: int) -> str:
    """Convert 0-based column index to A1 notation letter(s)."""
    result = ''
    col_idx += 1   # 1-based
    while col_idx:
        col_idx, rem = divmod(col_idx - 1, 26)
        result = chr(65 + rem) + result
    return result

# Metric labels in CSV_MAP → metric labels in the sheet
# If your sheet uses different labels, adjust this dict.
METRIC_ALIAS = {
    'BERTScore' : 'BERTScore',
    'ROUGE-1'   : 'ROUGE',      # adjust if there are separate rows per ROUGE variant
    'ROUGE-2'   : 'ROUGE',
    'ROUGE-L'   : 'ROUGE',
    'BARTScore' : 'BARTScore',
    'PPL'       : 'PPL',
    'SPICE'     : 'SPICE',
}

updates = []   # list of (A1_range, value) to batch-write

for metric_label, df in dfs.items():
    sheet_metric = METRIC_ALIAS.get(metric_label, metric_label)

    if sheet_metric not in metric_row_map:
        print(f'⚠  Metric "{sheet_metric}" not found in sheet rows — skipping {metric_label}')
        continue

    row_idx = metric_row_map[sheet_metric]

    for _, data_row in df.iterrows():
        cid = str(data_row['chart_id']).strip()
        cid = real_id_to_column.get(cid, cid)

        if cid not in chart_col_map:
            # Chart ID not in sheet header — skip silently
            continue

        for model in ['Qwen', 'ChartGemma']:
            if model not in chart_col_map[cid]:
                continue
            col_idx = chart_col_map[cid][model]
            a1 = f'{col_letter(col_idx)}{row_idx + 1}'   # A1 is 1-based
            value = float(data_row[model])
            updates.append({'range': a1, 'values': [[value]]})

print(f'Prepared {len(updates)} cell updates.')

Prepared 205 cell updates.


In [ ]:
# Batch-write in chunks to stay within Sheets API limits (300 ranges/request)
CHUNK = 300
total = len(updates)

for i in range(0, total, CHUNK):
    chunk = updates[i : i + CHUNK]
    ws.batch_update(chunk, value_input_option='RAW')
    print(f'  Written {min(i + CHUNK, total)}/{total} cells…')
    if i + CHUNK < total:
        time.sleep(1)   # brief pause to respect quota

  Written 205/205 cells…


check

In [ ]:
# Re-read the sheet and print the first filled metric row
refreshed = ws.get_all_values()

for metric, row_idx in metric_row_map.items():
    row = refreshed[row_idx]
    # Print first 10 non-empty cells after column C
    data_cells = [(i, v) for i, v in enumerate(row[3:], start=3) if v.strip()]
    if data_cells:
        print(f'{metric:15s} row {row_idx}: {data_cells[:6]}')

# ChartJudge

In [ ]:
import math

In [ ]:
CHARTJUDGE_FILES = {
    'ChartGemma': os.path.join(EVAL_ROOT, 'ChartJudge_Suite/ChartGemma_pointwise_scores.csv'),
    'Qwen':       os.path.join(EVAL_ROOT, 'ChartJudge_Suite/Qwen_pointwise_scores.csv'),
}

CHARTJUDGE_SCORE_COLS = {
    'factual_correctness_score' : 'FC*',
    'relevance_score'           : 'Relevance*',
    'fluency_score'             : 'Fluency*',
}


cj_updates = []

for model, filepath in CHARTJUDGE_FILES.items():
    df_cj = pd.read_csv(filepath)
    df_cj['img_id'] = df_cj['img_id'].astype(str).str.strip()

    for _, row in df_cj.iterrows():
        column = real_id_to_column.get(row['img_id'])
        if column is None:
            print(f"img_id {row['img_id']} not in ID→column map, skipping.")
            continue

        if column not in chart_col_map:
            print(f"column {column} not found in sheet header, skipping.")
            continue

        if model not in chart_col_map[column]:
            print(f"model '{model}' column not found for column {column}, skipping.")
            continue

        col_idx = chart_col_map[column][model]

        for score_col, sheet_metric in CHARTJUDGE_SCORE_COLS.items():
            if sheet_metric not in metric_row_map:
                print(f"Sheet row for '{sheet_metric}' not found, skipping.")
                continue

            row_idx = metric_row_map[sheet_metric]
            a1      = f'{col_letter(col_idx)}{row_idx + 1}'
            raw = row[score_col]
            try:
                value = float(raw)
                value = '' if (math.isnan(value) or math.isinf(value)) else value
            except (ValueError, TypeError):
                value = ''
            cj_updates.append({'range': a1, 'values': [[value]]})

print(f"Prepared {len(cj_updates)} ChartJudge cell updates.")

# Write to sheet
if cj_updates:
    ws.batch_update(cj_updates, value_input_option='RAW')
    print("ChartJudge scores written to sheet.")

# Questionnaire responses

In [ ]:
import re
import math
import pandas as pd
from collections import defaultdict

In [ ]:
FORM_EXCEL_PATH = '/content/drive/MyDrive/EvaltheEvaluators/evaluations/Responses.xlsx'
HUMAN_METRIC_MAP = {
    'more Factual'    : 'Factuality',
    'more Coherent'   : 'Coherence',
    'more Grammatical': 'Fluency',
}

def parse_col(col_name):
    m = re.search(r'\[(.+?)\](?:\s*(\d+))?', col_name)
    if not m:
        return None, None
    metric   = m.group(1).strip()
    chart_no = int(m.group(2)) if m.group(2) else 1
    return chart_no, metric

def preference_score_1_to_10(counts, model):
    """(wins + 0.5 × ties) / total, scaled from [0,1] → [1,10]."""
    total = counts['Qwen'] + counts['ChartGemma'] + counts['Tie']
    if total == 0:
        return ''
    raw = (counts[model] + 0.5 * counts['Tie']) / total
    return round(raw * 9 + 1, 4)

# parse
df_form     = pd.read_excel(FORM_EXCEL_PATH, sheet_name='Form responses 1')
answer_cols = [c for c in df_form.columns if 'Choose the best' in str(c)]
print(f"Loaded {len(df_form)} responses | {len(answer_cols)} answer columns")

votes = defaultdict(lambda: defaultdict(lambda: {'Qwen': 0, 'ChartGemma': 0, 'Tie': 0}))

for col in answer_cols:
    chart_no, metric = parse_col(col)
    if chart_no is None or metric not in HUMAN_METRIC_MAP:
        continue
    for val in df_form[col].dropna():
        val = str(val).strip()
        if   val == 'Description 1': votes[chart_no][metric]['Qwen'] += 1
        elif val == 'Description 2': votes[chart_no][metric]['ChartGemma'] += 1
        elif val == 'Tie':           votes[chart_no][metric]['Tie']        += 1

# preview
print(f"\n{'Chart':>6}  {'Metric':<18}  {'Qwen':>6}  {'ChartGemma':>10}  {'Tie':>4}  {'Qwen→1-10':>10}  {'CG→1-10':>10}")
print("-" * 80)
for chart_no in sorted(votes)[:3]:
    for metric, counts in votes[chart_no].items():
        q  = preference_score_1_to_10(counts, 'Qwen')
        cg = preference_score_1_to_10(counts, 'ChartGemma')
        print(f"{chart_no:>6}  {metric:<18}  {counts['Qwen']:>6}  {counts['ChartGemma']:>10}  {counts['Tie']:>4}  {q:>10}  {cg:>10}")

human_updates = []

for chart_no, metrics in votes.items():
    column = str(chart_no)
    if column not in chart_col_map:
        print(f"column {column} not in sheet header, skipping.")
        continue
    for metric_label, sheet_metric in HUMAN_METRIC_MAP.items():
        if metric_label not in metrics:
            continue
        if sheet_metric not in metric_row_map:
            print(f"  '{sheet_metric}' not found in sheet rows, skipping.")
            continue
        counts  = votes[chart_no][metric_label]
        row_idx = metric_row_map[sheet_metric]
        for model in ['Qwen', 'ChartGemma']:
            if model not in chart_col_map[column]:
                continue
            col_idx = chart_col_map[column][model]
            a1      = f'{col_letter(col_idx)}{row_idx + 1}'
            score   = preference_score_1_to_10(counts, model)
            human_updates.append({'range': a1, 'values': [[score]]})

print(f"\nPrepared {len(human_updates)} updates")
if human_updates:
    ws.batch_update(human_updates, value_input_option='RAW')
    print("  Human evaluation scores written to sheet.")

Loaded 7 responses | 63 answer columns

 Chart  Metric                Qwen  ChartGemma   Tie   Qwen→1-10     CG→1-10
--------------------------------------------------------------------------------
     1  more Factual             1           5     1      2.9286      8.0714
     1  more Coherent            2           4     1      4.2143      6.7857
     1  more Grammatical         1           5     1      2.9286      8.0714
     2  more Factual             2           4     1      4.2143      6.7857
     2  more Coherent            3           4     0      4.8571      6.1429
     2  more Grammatical         5           1     1      8.0714      2.9286
     3  more Factual             3           4     0      4.8571      6.1429
     3  more Coherent            2           5     0      3.5714      7.4286
     3  more Grammatical         1           3     3      4.2143      6.7857

Prepared 123 updates
  Human evaluation scores written to sheet.
